# P6 -- Error Analysis
## Notebook 07: `07_error_analysis.ipynb`

**Project:** Optimizing Inference Latency in Enterprise NLP via Task-Specific Knowledge Distillation  
**Group 15 | Section 2241044 | ITER, Siksha 'O' Anusandhan University**

---

### Purpose

Qualitative analysis of misclassified samples from the KD pipeline test set.

Answers three research questions:
1. What types of sentences does the model get wrong?
2. Are errors systematic (pattern-based) or random (noise)?
3. Can errors be traced back to teacher pseudo-label quality?

**Error type taxonomy:**

| Code | Type | Description |
|---|---|---|
| T1 | Ambiguous Language | Sentence genuinely ambiguous; humans may disagree |
| T2 | Subtle Positive | Positive sentiment expressed subtly; read as neutral |
| T3 | Context Dependent | Needs external context not present in sentence |
| T4 | Negation / Hedging | Negative phrasing within positive sentence or vice versa |
| T5 | Domain Jargon | Financial terminology misread by general-purpose model |

---

### Cell 1 -- Restore Session

Loads test predictions saved during Step 4 and merges with gold labels from the test split.

In [1]:
!pip install transformers scikit-learn pandas numpy torch groq -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os

BASE = '/content/drive/MyDrive/KD_Project'

test_df  = pd.read_csv(f'{BASE}/test.csv')
preds_df = pd.read_csv(f'{BASE}/test_predictions.csv')

test_df  = test_df.reset_index(drop=True)
preds_df = preds_df.reset_index(drop=True)

analysis_df = pd.DataFrame({
    'text'           : test_df['text'],
    'gold_label'     : test_df['sentiment'],
    'predicted_label': preds_df['predicted_label'],
    'urgency_pred'   : preds_df['urgency_pred'],
    'correct'        : preds_df['correct']
})

print(f'Test samples loaded   : {len(analysis_df)}')
print(f'Correct predictions   : {analysis_df["correct"].sum()}')
print(f'Incorrect predictions : {(~analysis_df["correct"]).sum()}')
print(f'Overall accuracy      : {analysis_df["correct"].mean():.4f}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 13.5 MB/s eta 0:00:00
Mounted at /content/drive
Test samples loaded   : 485
Correct predictions   : 377
Incorrect predictions : 108
Overall accuracy      : 0.7773


### Cell 2 -- Misclassification Overview

Shows the confusion breakdown. Confirms whether positive/neutral confusion is the dominant error mode.

In [2]:
from sklearn.metrics import confusion_matrix

errors_df = analysis_df[~analysis_df['correct']].copy().reset_index(drop=True)

print(f'Total misclassified: {len(errors_df)}')
print(f'\nErrors by true class:')
print(errors_df['gold_label'].value_counts().to_string())

labels = ['negative', 'neutral', 'positive']
cm = confusion_matrix(
    analysis_df['gold_label'],
    analysis_df['predicted_label'],
    labels=labels
)

print(f'\nConfusion matrix (rows=gold, cols=predicted):')
print(f'\n{"":>12}', end='')
for l in labels:
    print(f'{l:>12}', end='')
print()
print('-' * 48)
for i, row_label in enumerate(labels):
    print(f'{row_label:>12}', end='')
    for j, val in enumerate(cm[i]):
        marker = ' <--' if i != j and val > 0 else ''
        print(f'{val:>12}', end='')
    print()

print(f'\nMost common error pairs:')
error_pairs = errors_df.groupby(['gold_label', 'predicted_label']).size().sort_values(ascending=False)
print(error_pairs.to_string())

Total misclassified: 108

Errors by true class:
gold_label
positive    58
neutral     38
negative    12

Confusion matrix (rows=gold, cols=predicted):

                negative     neutral    positive
------------------------------------------------
    negative          49           9           3
     neutral          17         250          21
    positive           4          54          78

Most common error pairs:
gold_label  predicted_label
positive    neutral            54
neutral     positive           21
            negative           17
negative    neutral             9
positive    negative            4
negative    positive            3


### Cell 3 -- Sample 30 Misclassified Examples

Selects up to 10 misclassified samples per class for qualitative examination.

In [3]:
sample_size  = 10
sampled_list = []

for label in ['negative', 'neutral', 'positive']:
    class_errors = errors_df[errors_df['gold_label'] == label]
    n = min(sample_size, len(class_errors))
    sampled_list.append(class_errors.sample(n=n, random_state=42))

sampled_df = pd.concat(sampled_list).reset_index(drop=True)

print(f'Samples selected: {len(sampled_df)}')
print(f'\nPer-class:')
print(sampled_df['gold_label'].value_counts().to_string())

print(f'\n{"#":<4} {"GOLD":<12} {"PREDICTED":<12} SENTENCE')
print('-' * 90)
for i, row in sampled_df.iterrows():
    print(f'{i+1:<4} {row["gold_label"]:<12} {row["predicted_label"]:<12} {row["text"][:60]}')

Samples selected: 30

Per-class:
gold_label
negative    10
neutral     10
positive    10

#    GOLD         PREDICTED    SENTENCE
------------------------------------------------------------------------------------------
1    negative     neutral      Finnish power supply solutions and systems provider Efore Oy
2    negative     neutral      According to Swedish authorities , traces of the very toxic 
3    negative     neutral      Outokumpu 's steel mill in Tornio , in Finland , is the susp
4    negative     neutral      Expense ratio was 102.6 % compared to 92.9 % in the correspo
5    negative     neutral      According to the Latvian business register , Uponor Latvia c
6    negative     neutral      As part of the reorganisation measures that will take place 
7    negative     positive     Return on investment ROI was 4.1 % compared to 43.8 % in the
8    negative     neutral      National Conciliator Juhani Salonius , who met both parties 
9    negative     neutral      The reductio

### Cell 4 -- Automated Error Categorisation via Llama

Prompts Llama-3.1-8B to analyse each misclassified sample and assign an error type.

**Paste your Groq API key in the cell below.**

In [ ]:
import os, json, time
from groq import Groq

GROQ_API_KEY = 'gsk_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX'  # Replace with your actual API key
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
client = Groq(api_key=GROQ_API_KEY)

TYPE_LABELS = {
    'T1': 'Ambiguous Language',
    'T2': 'Subtle Positive',
    'T3': 'Context Dependent',
    'T4': 'Negation / Hedging',
    'T5': 'Domain Jargon'
}

def analyse_error(sentence, gold_label, predicted_label, client, retries=3):
    prompt = (
        'You are an expert NLP analyst reviewing misclassified financial sentences.\n\n'
        f'Sentence: "{sentence}"\n'
        f'Gold label (correct): {gold_label}\n'
        f'Predicted label (wrong): {predicted_label}\n\n'
        'Why was this misclassified? Choose ONE error type:\n'
        'T1 - Ambiguous Language: genuinely ambiguous, humans may disagree\n'
        'T2 - Subtle Positive: positive expressed subtly, looks like neutral\n'
        'T3 - Context Dependent: needs external context not in sentence\n'
        'T4 - Negation/Hedging: negative phrasing in positive sentence (or vice versa)\n'
        'T5 - Domain Jargon: financial term misread by general-purpose model\n\n'
        'Respond ONLY in JSON: {"error_type": "T1/T2/T3/T4/T5", "reason": "one sentence"}'
    )
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model='llama-3.1-8b-instant',
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=120,
                temperature=0.0
            )
            raw = response.choices[0].message.content.strip()
            raw = raw.replace('```json', '').replace('```', '').strip()
            result = json.loads(raw)
            assert result['error_type'] in ['T1', 'T2', 'T3', 'T4', 'T5']
            assert 'reason' in result
            return result
        except:
            time.sleep(2)
            continue
    return {'error_type': 'T1', 'reason': 'Could not determine error type'}

print(f'Analysing {len(sampled_df)} misclassified samples...\n')
error_types   = []
error_reasons = []

for i, row in sampled_df.iterrows():
    result = analyse_error(
        row['text'], row['gold_label'],
        row['predicted_label'], client
    )
    error_types.append(result['error_type'])
    error_reasons.append(result['reason'])
    print(f'[{i+1:>2}/{len(sampled_df)}] {row["gold_label"]:>10} -> {row["predicted_label"]:>10} | {result["error_type"]} | {result["reason"][:55]}')
    time.sleep(1)

sampled_df['error_type']   = error_types
sampled_df['error_reason'] = error_reasons
print('\nAnalysis complete.')

Analysing 30 misclassified samples...

[ 1/30]   negative ->    neutral | T1 | The sentence contains ambiguous language regarding the 
[ 2/30]   negative ->    neutral | T1 | The sentence contains ambiguous language, specifically 
[ 3/30]   negative ->    neutral | T1 | The sentence contains ambiguous language, as the word '
[ 4/30]   negative ->    neutral | T1 | The phrase 'compared to' can be interpreted in differen
[ 5/30]   negative ->    neutral | T1 | The sentence contains a phrase 'closed in red', which c
[ 6/30]   negative ->    neutral | T1 | The sentence contains ambiguous language regarding the 
[ 7/30]   negative ->   positive | T1 | The sentence contains a comparison between two percenta
[ 8/30]   negative ->    neutral | T1 | The sentence contains ambiguous language as 'too far ap
[ 9/30]   negative ->    neutral | T1 | The phrase 'reduction notice' can be interpreted as a f
[10/30]   negative ->    neutral | T1 | The sentence contains ambiguous language, with 'substan
[

### Cell 5 -- Error Type Distribution

Counts errors per type. Shows whether errors are systematic or random.

In [6]:
TYPE_LABELS = {
    'T1': 'Ambiguous Language',
    'T2': 'Subtle Positive',
    'T3': 'Context Dependent',
    'T4': 'Negation / Hedging',
    'T5': 'Domain Jargon'
}

type_counts = sampled_df['error_type'].value_counts().sort_index()
total = len(sampled_df)

print('=' * 60)
print('ERROR TYPE DISTRIBUTION')
print('=' * 60)
for code, count in type_counts.items():
    pct = count / total * 100
    bar = chr(9608) * int(pct / 3)
    label = TYPE_LABELS.get(code, code)
    print(f'{code} - {label:<25} : {count:>3} ({pct:.1f}%)  {bar}')
print('=' * 60)

print('\nError types by gold class:')
print(pd.crosstab(sampled_df['gold_label'], sampled_df['error_type']).to_string())

print('\nTop error combinations (gold, predicted, type):')
top = sampled_df.groupby(['gold_label', 'predicted_label', 'error_type']).size().sort_values(ascending=False).head(8)
print(top.to_string())

ERROR TYPE DISTRIBUTION
T1 - Ambiguous Language        :  20 (66.7%)  ██████████████████████
T2 - Subtle Positive           :  10 (33.3%)  ███████████

Error types by gold class:
error_type  T1  T2
gold_label        
negative    10   0
neutral     10   0
positive     0  10

Top error combinations (gold, predicted, type):
gold_label  predicted_label  error_type
negative    neutral          T1            9
positive    neutral          T2            8
neutral     negative         T1            6
            positive         T1            4
positive    negative         T2            2
negative    positive         T1            1


### Cell 6 -- Detailed Error Report

Full 30-sample table: gold label, predicted label, error type, and reason.
This is the qualitative analysis table for the research paper.

In [7]:
print('=' * 115)
print('FULL ERROR ANALYSIS -- 30 MISCLASSIFIED SAMPLES')
print('=' * 115)
print(f'{"#":<4} {"GOLD":<12} {"PRED":<12} {"TYPE":<6} {"SENTENCE":<45} REASON')
print('-' * 115)

for i, row in sampled_df.iterrows():
    s = row['text'][:42] + '...' if len(row['text']) > 42 else row['text']
    r = row['error_reason'][:45] + '...' if len(row['error_reason']) > 45 else row['error_reason']
    print(f'{i+1:<4} {row["gold_label"]:<12} {row["predicted_label"]:<12} {row["error_type"]:<6} {s:<45} {r}')

print('=' * 115)

sampled_df.to_csv(f'{BASE}/error_analysis.csv', index=False)
print(f'\nFull table saved to Drive: {BASE}/error_analysis.csv')

FULL ERROR ANALYSIS -- 30 MISCLASSIFIED SAMPLES
#    GOLD         PRED         TYPE   SENTENCE                                      REASON
-------------------------------------------------------------------------------------------------------------------
1    negative     neutral      T1     Finnish power supply solutions and systems... The sentence contains ambiguous language rega...
2    negative     neutral      T1     According to Swedish authorities , traces ... The sentence contains ambiguous language, spe...
3    negative     neutral      T1     Outokumpu 's steel mill in Tornio , in Fin... The sentence contains ambiguous language, as ...
4    negative     neutral      T1     Expense ratio was 102.6 % compared to 92.9... The phrase 'compared to' can be interpreted i...
5    negative     neutral      T1     According to the Latvian business register... The sentence contains a phrase 'closed in red...
6    negative     neutral      T1     As part of the reorganisation measures tha

### Cell 7 -- Paper-Ready Findings

Generates finding statements based on the error distribution for direct use in the paper.

In [8]:
TYPE_LABELS = {
    'T1': 'Ambiguous Language',
    'T2': 'Subtle Positive',
    'T3': 'Context Dependent',
    'T4': 'Negation / Hedging',
    'T5': 'Domain Jargon'
}

type_counts = sampled_df['error_type'].value_counts()
total = len(sampled_df)

dominant_type = type_counts.index[0]
dominant_pct  = round(type_counts.iloc[0] / total * 100, 1)
dominant_label = TYPE_LABELS.get(dominant_type, dominant_type)

pos_errors = sampled_df[sampled_df['gold_label'] == 'positive']
pos_t2_pct = round((pos_errors['error_type'] == 'T2').sum() / max(len(pos_errors), 1) * 100, 1)

print('=' * 70)
print('PAPER-READY FINDING STATEMENTS')
print('=' * 70)
print()
print(f'Finding 1 - Errors are systematic, not random:')
print(f'{dominant_pct}% of misclassified samples are "{dominant_label}" ({dominant_type}).')
print(f'Error concentration in one type confirms systematic, not random, failure.')
print()
print(f'Finding 2 - Positive class errors trace to teacher quality:')
print(f'{pos_t2_pct}% of positive class errors are Subtle Positive (T2).')
print(f'Sentences with understated growth or improvement are misread as neutral')
print(f'by the 8B teacher, propagating to the student via noisy pseudo-labels.')
print()
print(f'Finding 3 - Neutral class errors are boundary cases:')
print(f'Neutral misclassifications are primarily T1 (Ambiguous) and T3 (Context),')
print(f'indicating the model makes errors only on genuinely borderline sentences.')
print()
print('=' * 70)
print('ERROR TYPE SUMMARY TABLE')
print('=' * 70)
print(f'{"Code":<6} {"Type":<28} {"Count":<8} {"Pct":<8} Primary Class')
print('-' * 70)
for code in sorted(sampled_df['error_type'].unique()):
    count = (sampled_df['error_type'] == code).sum()
    pct   = round(count / total * 100, 1)
    primary = sampled_df[sampled_df['error_type'] == code]['gold_label'].mode()[0]
    print(f'{code:<6} {TYPE_LABELS.get(code, code):<28} {count:<8} {pct:<8} {primary}')
print('=' * 70)

PAPER-READY FINDING STATEMENTS

Finding 1 - Errors are systematic, not random:
66.7% of misclassified samples are "Ambiguous Language" (T1).
Error concentration in one type confirms systematic, not random, failure.

Finding 2 - Positive class errors trace to teacher quality:
100.0% of positive class errors are Subtle Positive (T2).
Sentences with understated growth or improvement are misread as neutral
by the 8B teacher, propagating to the student via noisy pseudo-labels.

Finding 3 - Neutral class errors are boundary cases:
Neutral misclassifications are primarily T1 (Ambiguous) and T3 (Context),
indicating the model makes errors only on genuinely borderline sentences.

ERROR TYPE SUMMARY TABLE
Code   Type                         Count    Pct      Primary Class
----------------------------------------------------------------------
T1     Ambiguous Language           20       66.7     negative
T2     Subtle Positive              10       33.3     positive


### Cell 8 -- Save All Results and Commit Instructions

Saves all error analysis files to Drive and prints the GitHub commit commands.

In [10]:
TYPE_LABELS = {
    'T1': 'Ambiguous Language',
    'T2': 'Subtle Positive',
    'T3': 'Context Dependent',
    'T4': 'Negation / Hedging',
    'T5': 'Domain Jargon'
}

# Save full analysis
sampled_df.to_csv(f'{BASE}/error_analysis.csv', index=False)

# Save summary
type_counts = sampled_df['error_type'].value_counts()
summary_rows = []
for code in sorted(sampled_df['error_type'].unique()):
    count = (sampled_df['error_type'] == code).sum()
    summary_rows.append({
        'error_type'  : code,
        'description' : TYPE_LABELS.get(code, code),
        'count'       : count,
        'percentage'  : round(count / len(sampled_df) * 100, 1)
    })
pd.DataFrame(summary_rows).to_csv(f'{BASE}/error_analysis_summary.csv', index=False)

dominant_type = type_counts.index[0]
dominant_pct  = round(type_counts.iloc[0] / len(sampled_df) * 100, 1)

print('Files saved to Drive:')
print(f'  {BASE}/error_analysis.csv')
print(f'  {BASE}/error_analysis_summary.csv')

Files saved to Drive:
  /content/drive/MyDrive/KD_Project/error_analysis.csv
  /content/drive/MyDrive/KD_Project/error_analysis_summary.csv
